### Webpage summarizer

In [10]:
# imports

import os
from dotenv import load_dotenv
from importlib import reload
# from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


In [11]:
# Check the key

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


openai = OpenAI()
message = "Hello, GPT! This is MK"
messages = [{"role": "user", "content": message}]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

API key found and looks good so far!


'Hello, MK! How can I assist you today?'

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time
import os 

class Website:
    def __init__(self, url, driver_path=None, wait_time=3):
        self.url = url
        self.wait_time = wait_time

        # Headless Chrome settings
        options = Options()
        # options.add_argument("--headless")  
        # Headless mode runs the browser in the background (invisible).
        # However, some websites (like openai.com) block headless browsers.
        # So if this line is active, the page may not load correctly and you may not get the full content.
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920x1080")

        # Driver path
        if driver_path:
            service = Service(executable_path=driver_path)
        else:
            service = Service() 

        # Start browser
        driver = webdriver.Chrome(service=service, options=options)
        driver.get(url)

        # Wait for the loading page
        time.sleep(self.wait_time)

        # Take page source
        html = driver.page_source
        driver.quit()

        # Analysis with BeautifulSoup 
        soup = BeautifulSoup(html, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"

        # Clean irrelevant tags
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()

        self.text = soup.body.get_text(separator="\n", strip=True)
# # Test the new function with the AEMO URL
# url = "https://www.aemo.com.au/newsroom/media-release/renewables-supply-more-than-half-of-quarterly-energy-supply"
# website_content = fetch_website_contents_selenium(url)

# if website_content:
#     print("Successfully fetched content!")
#     print(website_content[:500]) # Print first 500 chars
    
#     # You can now proceed to summarize this content using the existing summarize function logic
#     # but passing the content directly instead of fetching it again.

In [5]:
system_prompt = "You are an assitant helping to summarize the content on the provided webpages"
user_prompt_prefix = """This is the fetched content of the webpage, please return a summary of the content.  For the summary, in this format: " \
1) start with a brief summary of the webpage content, no more than 100 words. 
2) List key points, each key point starts with a short title, followed by a bit more details. Each key point no more than 20 words.  
""" 



In [7]:
# Selenium implementation to handle JavaScript-rendered websites
# Improved with undetected-chromedriver to bypass Cloudflare
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import time
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def fetch_website_contents_selenium(url):
    print(f"Fetching content from {url} using undetected-chromedriver...")
    driver = None
    
    try:
        options = uc.ChromeOptions()
        # Headless mode is often detected by Cloudflare. 
        # We'll run in headed mode (visible browser) which is more reliable for local scripts.
        # options.add_argument("--headless") 
        options.add_argument("--disable-gpu")
        
        # Initialize undetected-chromedriver
        # Manually specifying the version_main to match the installed Chrome version
        driver = uc.Chrome(options=options, use_subprocess=True, version_main=144)
        
        driver.get(url)
        
        # Wait for Cloudflare challenge to potentially clear or page to load
        print("Waiting for page to load...")
        time.sleep(5) 
        
        # Get body text
        body_content = driver.find_element(By.TAG_NAME, "body").text
        return body_content
        
    except Exception as e:
        print(f"Error: {e}")
        return None
    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass



In [12]:
url = "https://www.aemo.com.au/newsroom/media-release/renewables-supply-more-than-half-of-quarterly-energy-supply" 
website = fetch_website_contents_selenium(url)
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages_for(website = website))

# Step 4: print the result

display(Markdown(response.choices[0].message.content))

Fetching content from https://www.aemo.com.au/newsroom/media-release/renewables-supply-more-than-half-of-quarterly-energy-supply using undetected-chromedriver...
Waiting for page to load...


1) Summary
AEMO’s Q4 2025 Quarterly Energy Dynamics shows renewables and storage delivering more than half of NEM energy for the quarter, driving lower wholesale prices and a shift away from coal and gas, with rooftop solar and regional markets reflecting the trend.

2) Milestone: Renewables exceed 50% of NEM energy – renewables and storage supply over half the quarter.

3) Price drop: Wholesale prices averaged $50/MWh; down 44% vs Q4 2024, 43% vs Q3 2025.

4) Growth in renewables: Wind +29%, solar +15%, battery discharge to 268 MW; 3,796 MW more capacity since late-2024.

5) Coal and gas declines: Coal down 4.6%; gas down 27% to their lowest since Q4 2000.

6) Rooftop solar and demand: Rooftop solar at 4,407 MW (+8.7%); daytime demand lower; new minimum operational demand records.

7) East Coast Gas Market: Gas prices $12.68/GJ; demand down 3%; 2.3 PJ more domestic gas from LNG export drop.

8) Western Australia momentum: WA renewables 52.4%; output peaks 91.1%; price down to $69.55/MWh.

9) Domestic gas market: Consumption down 9.3% to 94.5 PJ; production down 1.7% to 102 PJ.